# NB15c — V1 Pine Export (Build 2)

**Datum:** 2026-05-28
**Anker:** [ANN-018 Execution-Lock + 3-Layer Skeleton](../docs/decisions/ANN-018-decision-assisted-architecture-multi-timeframe-dashboard.md), [ANN-016 FX as Product](../docs/decisions/ANN-016-fx-as-reference-blueprint-industrialization-first.md), Nicos Build-2-Direktive 2026-05-28 (Pfad B: data-driven feature selection).

**Was dieses Notebook tut (in genau dieser Reihenfolge):**
1. Lade Trainings-Daten + trainiere NB14f-v2 production model (seed=7, deterministic) ODER reuse cached booster
2. Extract VAL probability → 3 Cutoffs (production_cluster + Rank-1 + Rank-2)
3. `extract_feature_usage(booster)` → `results/nb15c/feature_usage.json` (Path-B Feature-Selection)
4. Render `pine_features.render_feature_engine(used_features)` — FAIL CLEAR wenn ein Feature kein Pine-Snippet hat
5. `lgbm_to_pine_cascade(booster, used_features)` → Pine-Cascade-Block
6. Patch `deploy_pine/pace_algo_v1_skeleton.pine` → `deploy_pine/pace_algo_v1.pine` (Placeholder + Cutoffs ersetzt)
7. `estimate_pine_ops(patched_file)` → `results/nb15c/pine_budget.json` (Warning wenn > 70%)
8. Bit-exact (Stage B): `booster.predict()` vs `python_reimplementation()` auf TEST-Samples → `results/nb15c/bit_exact.json` (atol 1e-5)
9. Auto-Push aller Outputs zu GitHub

**Keine UI-/Backtest-/Feature-Engineering-Arbeit hier.** ANN-018 Execution-Lock.

**Output für Nico:** `deploy_pine/pace_algo_v1.pine` — direkt pasteable in TradingView Pine Editor. Build-3-Live-Test.


## Section 0 — Setup + Module Sync

In [ ]:
import os, sys, subprocess, gc, json, importlib
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1',
                    'https://github.com/ecoNC/pace-algo.git', '/tmp/pace-algo'], check=True)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router {PROJECT_ROOT}/core/export',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/deploy_pine {PROJECT_ROOT}/', shell=True)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    print('Core + deploy_pine synced.')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

import numpy as np
RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'], text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb15c_{RUN_TIMESTAMP}_{GIT_COMMIT}'
print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'pyarrow>=15.0'], capture_output=True)

import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import FX_TRAIN_SYMBOLS, TRAIN_END, VAL_END
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS
from core.analysis.probability_diagnostic import extract_premium_cluster
from core.export.pine_codegen import (
    lgbm_to_pine_cascade, extract_feature_usage, estimate_pine_ops,
    bit_exact_check, python_reimplementation,
)
from core.export.pine_features import render_feature_engine, FEATURE_REGISTRY

TF = '5m'
R_VALUE = 1.5
PRODUCTION_SEED = 7
MIN_CLUSTER_SIZE_PCT = 0.5
DECIMAL_PLACES = 4

OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb15c'
(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Production seed:   {PRODUCTION_SEED}')
print(f'Output dir:        {OUTPUT_DIR}')
print(f'Features in registry: {len(FEATURE_REGISTRY)}')


## Section 1 — Load Training Pool

NB14f-v2-Pattern: FX_TRAIN_SYMBOLS × 5m extended files. Falls `_extended.parquet`
fehlt: fail clear (NB14f Section 1 Auto-Build hat das schon mal geliefert).

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = cfg.DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = cfg.DATA_PROCESSED

def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if 'hit_bar_offset' not in df.columns:
        df['hit_bar_offset'] = 24
    return df

missing = [s for s in FX_TRAIN_SYMBOLS if load_ext(s, TF) is None]
if missing:
    raise SystemExit(
        f'Missing _extended for: {missing}. '
        f'Run NB14f Section 1 Auto-Build first to (re)generate them.'
    )

frames = []
for s in FX_TRAIN_SYMBOLS:
    d = load_ext(s, TF)
    d['symbol'] = s
    frames.append(d.astype({c: 'float32' for c in d.select_dtypes(include=['float64']).columns}))
pool = pd.concat(frames, axis=0).sort_index()
del frames; gc.collect()

probe = load_ext(FX_TRAIN_SYMBOLS[0], TF)
FEATURE_COLS = [c for c in probe.columns if c not in NON_FEATURE_COLS and c != 'symbol']
print(f'Features in extended data: {len(FEATURE_COLS)}')

pool_c = pool.dropna(subset=FEATURE_COLS + ['label'])
train_df, val_df, test_df = walk_forward_split(pool_c, TRAIN_END, VAL_END)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = binary_label_for_long(train_df['label']).values
X_val   = val_df[FEATURE_COLS].values.astype(np.float32)
y_val   = binary_label_for_long(val_df['label']).values
X_test  = test_df[FEATURE_COLS].values.astype(np.float32)
del pool, pool_c
gc.collect()


## Section 2 — Train Production Booster (seed=7, deterministic)

NB14f-v2-Pattern, but lock to seed=7 (BEST_SEED from NB14f-v2 verdict).

In [ ]:
BASE_PARAMS = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 100,
    'lambda_l2': 1.0, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 5, 'is_unbalance': True,
    'verbose': -1, 'n_jobs': -1,
    'seed': PRODUCTION_SEED,
    'deterministic': True,
}

td = lgb.Dataset(X_train, label=y_train, feature_name=FEATURE_COLS)
vd = lgb.Dataset(X_val, label=y_val, reference=td, feature_name=FEATURE_COLS)
booster = lgb.train(BASE_PARAMS, td, valid_sets=[td, vd], valid_names=['train','val'],
                    callbacks=[lgb.early_stopping(10, verbose=False), lgb.log_evaluation(period=0)])

n_trees = booster.num_trees()
print(f'Booster trained: {n_trees} trees')

# Persist booster locally + in Drive
MODEL_DIR = Path(PROJECT_ROOT) / 'artifacts' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / f'fx_v1_lgbm_seed{PRODUCTION_SEED}_{RUN_DATE}.txt'
booster.save_model(str(model_path))
print(f'Model saved: {model_path}')


## Section 3 — Extract Cutoffs (3 Tier Levels)

Per NB14f-v2 + Sibling-Skeleton 3-Tier-Structure:
- PREMIUM = `production_cluster_value` (höchster qualifying cluster, ANN-013/14)
- HIGH = Rank-1-Cluster (zweithöchster)
- STANDARD = Rank-2-Cluster (dritthöchster)

Wenn das Modell weniger als 3 qualifying clusters hat: fall back auf Quantile.

In [ ]:
proba_val = booster.predict(X_val)
cluster = extract_premium_cluster(
    proba_val, decimal_places=DECIMAL_PLACES,
    min_cluster_size_pct=MIN_CLUSTER_SIZE_PCT,
)

if not cluster['success']:
    raise SystemExit(f'Cluster extraction failed: {cluster.get("error")}')

qualifying = cluster['all_qualifying_clusters']  # sorted high→low
print(f'Qualifying clusters: {len(qualifying)}')
for i, c in enumerate(qualifying[:5]):
    print(f'  rank {i}: value={c["value"]:.4f}  size={c["count"]}  pct={c["pct"]:.2f}%')

# Tier cutoffs from qualifying clusters (deterministic from VAL)
CUTOFF_PREMIUM  = float(qualifying[0]['value'])
CUTOFF_HIGH     = float(qualifying[1]['value']) if len(qualifying) > 1 else CUTOFF_PREMIUM
CUTOFF_STANDARD = float(qualifying[2]['value']) if len(qualifying) > 2 else CUTOFF_HIGH

# Sanity: monotone descending
if not (CUTOFF_PREMIUM >= CUTOFF_HIGH >= CUTOFF_STANDARD):
    raise SystemExit(f'Cutoffs not monotone: P={CUTOFF_PREMIUM}, H={CUTOFF_HIGH}, S={CUTOFF_STANDARD}')

print()
print(f'CUTOFF_PREMIUM  = {CUTOFF_PREMIUM:.4f}')
print(f'CUTOFF_HIGH     = {CUTOFF_HIGH:.4f}')
print(f'CUTOFF_STANDARD = {CUTOFF_STANDARD:.4f}')


## Section 4 — Feature Usage Report (Path B)

Pfad B: nur Features die der Booster wirklich referenziert. Output: `results/nb15c/feature_usage.json`.

In [ ]:
usage = extract_feature_usage(booster, FEATURE_COLS)
print(f'Features in training set: {len(FEATURE_COLS)}')
print(f'Features actually used:   {usage["n_features_referenced"]}')
print(f'Features unused (SHAP=0): {len(usage["unused_features"])}')
print()
print('Top 10 by split_count:')
for d in usage['details'][:10]:
    print(f'  {d["feature_name"]:30s} splits={d["split_count"]:3d}  trees={d["used_in_tree_count"]:3d}  avg_gain={d["avg_gain"]:.4f}')

print()
print(f'Unused features: {usage["unused_features"]}')

# Persist
usage_path = OUTPUT_DIR / 'feature_usage.json'
with open(usage_path, 'w') as f:
    json.dump({
        'experiment_id': EXPERIMENT_ID,
        'run_date':      RUN_DATE,
        'production_seed': PRODUCTION_SEED,
        **usage,
    }, f, indent=2)
print(f'Saved: {usage_path}')


## Section 5 — Render Feature Engine for Used Features

`render_feature_engine()` returns `missing=[]` only if EVERY used feature has a Pine snippet.

In [ ]:
used = usage['used_features']
engine = render_feature_engine(used)

if engine['missing']:
    raise SystemExit(
        f'Features without Pine snippet: {engine["missing"]}\n'
        f'Add them to core/export/pine_features.py FEATURE_REGISTRY.'
    )

print(f'Feature engine rendered for {len(used)} features.')
print(f'HTF header needed: {engine["htf"] != ""}')
print(f'Helpers: {len(engine["helpers"])} chars')
print(f'Features: {len(engine["features"])} chars')


## Section 6 — Generate Pine Tree Cascade

In [ ]:
pine_cascade = lgbm_to_pine_cascade(booster, used)
print(f'Cascade: {pine_cascade.count(chr(10))} lines, {len(pine_cascade)} bytes')
print()
print('First 6 lines:')
for line in pine_cascade.split('\n')[:6]:
    print(f'  {line}')
print('  ...')
print('Last 3 lines:')
for line in pine_cascade.split('\n')[-4:-1]:
    print(f'  {line}')


## Section 7 — Patch Skeleton → `deploy_pine/pace_algo_v1.pine`

In [ ]:
SKELETON_PATH = Path(PROJECT_ROOT) / 'deploy_pine' / 'pace_algo_v1_skeleton.pine'
OUTPUT_PINE = Path(PROJECT_ROOT) / 'deploy_pine' / 'pace_algo_v1.pine'

skeleton = SKELETON_PATH.read_text(encoding='utf-8')

# --- Build replacement block ---
# Order in Pine: helpers → htf → features → cascade function → probability call
replacement_block = '''// === BUILD 2 INSERT: V1 Feature Engine + Tree Cascade ===
// Source: NB15c {EXPERIMENT_ID}, booster seed={SEED}, {NTREES} trees
// Auto-generated by core/export/pine_codegen.py + pine_features.py
// DO NOT EDIT — regenerate via notebooks/15c_v1_pine_export.ipynb

{HELPERS}
{HTF}
{FEATURES}

{CASCADE}'''.format(
    EXPERIMENT_ID=EXPERIMENT_ID,
    SEED=PRODUCTION_SEED,
    NTREES=n_trees,
    HELPERS=engine['helpers'].rstrip(),
    HTF=engine['htf'].rstrip(),
    FEATURES=engine['features'].rstrip(),
    CASCADE=pine_cascade.rstrip(),
)

# --- Find + replace placeholder function (Skeleton lines 48-58) ---
PLACEHOLDER_START = 'f_signal_probability_placeholder() =>'
PLACEHOLDER_END_MARKER = ': 0.50'
ps = skeleton.index(PLACEHOLDER_START)
pe = skeleton.index(PLACEHOLDER_END_MARKER, ps) + len(PLACEHOLDER_END_MARKER)
patched = skeleton[:ps] + replacement_block + skeleton[pe:]

# --- Replace probability call ---
old_call = 'probability = f_signal_probability_placeholder()'
new_call = f'probability = f_pace_algo_v1_probability({engine["feature_arg_list"]})'
if old_call not in patched:
    raise SystemExit(f'Could not find probability call: {old_call!r}')
patched = patched.replace(old_call, new_call, 1)

# --- Replace cutoffs (Skeleton lines 64-66) ---
patched = patched.replace(
    'var float CUTOFF_STANDARD = 0.50',
    f'var float CUTOFF_STANDARD = {CUTOFF_STANDARD:.4f}', 1,
)
patched = patched.replace(
    'var float CUTOFF_HIGH     = 0.55',
    f'var float CUTOFF_HIGH     = {CUTOFF_HIGH:.4f}', 1,
)
patched = patched.replace(
    'var float CUTOFF_PREMIUM  = 0.65',
    f'var float CUTOFF_PREMIUM  = {CUTOFF_PREMIUM:.4f}', 1,
)

# --- Sanity: no placeholder strings left ---
LEFTOVERS = ['f_signal_probability_placeholder', 'CUTOFF_STANDARD = 0.50',
             'CUTOFF_HIGH     = 0.55', 'CUTOFF_PREMIUM  = 0.65']
remaining = [l for l in LEFTOVERS if l in patched]
if remaining:
    raise SystemExit(f'Placeholders still present after patch: {remaining}')

OUTPUT_PINE.write_text(patched, encoding='utf-8')
print(f'Wrote: {OUTPUT_PINE}')
print(f'Patched file: {patched.count(chr(10))} lines, {len(patched)} bytes')


## Section 8 — Pine Budget Estimate + Warning if >70%

In [ ]:
budget_report = estimate_pine_ops(patched, pine_budget_per_bar=5000, request_security_budget=40)

print(f"Pine budget estimate (final file):")
print(f"  ops/bar (heuristic):   {budget_report['ops_estimate']}  ({budget_report['ops_pct_of_budget']:.1%} of {budget_report['pine_budget_per_bar']})")
print(f"  request.security:      {budget_report['request_security_calls']}  ({budget_report['request_security_pct_of_budget']:.1%} of {budget_report['request_security_budget']})")
print(f"  total lines:           {budget_report['n_lines']}")
print(f"  total bytes:           {budget_report['n_bytes']}")
print(f"  ta.* calls:            {budget_report['ta_calls']}")
print(f"  math.* calls:          {budget_report['math_calls']}")
print(f"  passed_budget_check:   {budget_report['passed_budget_check']}")

if budget_report['warnings']:
    print()
    print('⚠️  WARNINGS:')
    for w in budget_report['warnings']:
        print(f'   • {w}')

# Persist
budget_path = OUTPUT_DIR / 'pine_budget.json'
with open(budget_path, 'w') as f:
    json.dump({
        'experiment_id': EXPERIMENT_ID,
        **budget_report,
    }, f, indent=2)
print(f'Saved: {budget_path}')


## Section 9 — Bit-Exact Validation (Stage B)

Python `booster.predict(X_test)` vs `python_reimplementation()` der Cascade.
Stage A war Mini-LGBM lokal (tests/test_pine_codegen.py).
Stage C (Pine actual vs Python) wird in TradingView visuell beim Build-3-Test geprüft.

In [ ]:
N_SAMPLES = 10_000
sample_idx = np.random.choice(len(X_test), min(N_SAMPLES, len(X_test)), replace=False)
X_check = X_test[sample_idx]

check = bit_exact_check(booster, FEATURE_COLS, X_check, atol=1e-5)
print(f"Bit-exact (Stage B):")
print(f"  n_samples:    {check['n_samples']}")
print(f"  max_abs_diff: {check['max_abs_diff']:.2e}")
print(f"  rmse:         {check['rmse']:.2e}")
print(f"  atol:         {check['atol']:.0e}")
print(f"  passed:       {check['passed']}")

# CRITICAL: feature index in cascade must reference FEATURE_COLS, not just used.
# We pass FEATURE_COLS (full set) into python_reimplementation — that's how the
# booster's split_feature indices are interpreted.

# Persist
bitexact_path = OUTPUT_DIR / 'bit_exact.json'
with open(bitexact_path, 'w') as f:
    json.dump({
        'experiment_id': EXPERIMENT_ID,
        'stage':         'B (booster.predict vs python_reimplementation)',
        **check,
    }, f, indent=2)
print(f'Saved: {bitexact_path}')

if not check['passed']:
    raise SystemExit(f'BIT-EXACT FAILED: {check}')


## Section 10 — Production Snapshot + Auto-Push

In [ ]:
snapshot = {
    'experiment_id':         EXPERIMENT_ID,
    'run_date':              RUN_DATE,
    'git_commit':            GIT_COMMIT,
    'production_seed':       PRODUCTION_SEED,
    'n_trees':               n_trees,
    'feature_cols_training': FEATURE_COLS,
    'feature_cols_used':     used,
    'feature_cols_unused':   usage['unused_features'],
    'cutoffs': {
        'CUTOFF_STANDARD': CUTOFF_STANDARD,
        'CUTOFF_HIGH':     CUTOFF_HIGH,
        'CUTOFF_PREMIUM':  CUTOFF_PREMIUM,
    },
    'pine_output_path':      str(OUTPUT_PINE.relative_to(PROJECT_ROOT)),
    'pine_budget':           budget_report,
    'bit_exact':             check,
    'base_params':           BASE_PARAMS,
}
snap_path = OUTPUT_DIR / 'snapshot.json'
with open(snap_path, 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)
print(f'Snapshot: {snap_path}')

# --- Auto-push (NB14f-Pattern) ---
import shutil
if not IN_COLAB:
    print('Local run — skip auto-push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GH_TOKEN = None

    if not GH_TOKEN:
        print('⚠️  GITHUB_TOKEN nicht gefunden — Outputs bleiben im Drive.')
    else:
        TMP_REPO = Path('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email',
                        'ecoNC@users.noreply.github.com'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--autostash', '--quiet',
                        'origin', 'main'], check=True)
        print('Pulled latest from origin/main')

        copied = []
        # results/nb15c/* outputs
        for f in sorted(OUTPUT_DIR.glob('*.json')):
            rel = f.relative_to(Path(PROJECT_ROOT))
            dest = TMP_REPO / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(str(rel))
        # The patched Pine file
        dest_pine = TMP_REPO / 'deploy_pine' / 'pace_algo_v1.pine'
        dest_pine.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(OUTPUT_PINE, dest_pine)
        copied.append('deploy_pine/pace_algo_v1.pine')
        print(f'Copied {len(copied)} files')

        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/', 'deploy_pine/'], check=True)
        r_status = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'],
                                  capture_output=True, text=True)
        if not r_status.stdout.strip():
            print('Nothing to commit.')
        else:
            ops_pct = budget_report['ops_pct_of_budget']
            msg = (f'NB15c V1 Pine Export {RUN_DATE} (seed={PRODUCTION_SEED}, '
                   f'{n_trees} trees, {len(used)} used features, '
                   f'ops={budget_report["ops_estimate"]}/{budget_report["pine_budget_per_bar"]})')
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                 capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'PUSHED as ecoNC ({sha})')
            print(f'  https://github.com/ecoNC/pace-algo/commit/{sha}')
        shutil.rmtree(TMP_REPO)

print()
print('=' * 60)
print('BUILD 2 COMPLETE.')
print('=' * 60)
print(f'Pine file ready: deploy_pine/pace_algo_v1.pine')
print(f'  → Paste in TradingView Pine Editor and add to chart')
print(f'  → Test on EURUSD/GBPUSD/USDJPY/AUDUSD/USDCAD/NZDUSD 5m')
print(f'  → USDCHF: läuft, ist aber low-confidence pair (NB14f-v2 verdict)')
